In [1]:
import sys
# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [2]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig, DEFAULT_INMEMORYGRAPH_CONFIG
from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig, DEFAULT_INMEMORYKV_CONFIG

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.memorize_pipeline import MemPipelineConfig, LLMExtractorConfig, LLMUpdatorConfig

from src.utils import Logger

/home/dzigen/Desktop/PersonalAI/pai_venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


#### 1. Задаём конфигурацию графа знаний

In [3]:
# CONFIG FOR REMOTE STORAGE

remote_kg_config = RemoteKnowledgeGraphConfig(
    graph_struct_config=GraphModelConfig(driver_config=GraphDriverConfig(
        db_vendor='neo4j', db_config=GraphDBConnectionConfig(uri="bolt://localhost:7687", params={'user': "neo4j", 'pwd': 'password', 'db_name': 'testing'}))), # TO CHANGE
    
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_nodes/testing', db_name='vectorized_nodes', need_to_clear=True)), # TO CHANGE
        tripletsdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_triplets/testing', db_name='vectorized_triplets', need_to_clear=True)), # TO CHANGE
        embedder_config=EmbedderModelConfig(model_name_or_path='../../models/intfloat/multilingual-e5-small')),
    
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method='astar', # TO CHANGE
            retriever_config=AStarGraphSearchConfig(), # TO CHANGE
            cache_config=KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)), #KeyValueDriverConfig(db_vendor='aerospike', db_config=KVDBConnectionConfig(host='aerospikelservice', port=3000))),
        answer_generator_config=QALLMGeneratorConfig(lang='eng')),
    
    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(),
        updator_config=LLMUpdatorConfig()),
    
    log=Logger('log/main'))

In [3]:
# CONFIG FOR IN-MEMORY STORAGE

inmemory_kg_config = RemoteKnowledgeGraphConfig(
    
    graph_struct_config=GraphModelConfig(driver_config=GraphDriverConfig(
        db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)), # TO CHANGE
    
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_nodes/testing', db_name='vectorized_nodes', need_to_clear=True)), # TO CHANGE
        tripletsdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_triplets/testing', db_name='vectorized_triplets', need_to_clear=True)), # TO CHANGE
        embedder_config=EmbedderModelConfig(model_name_or_path='intfloat/multilingual-e5-small')),
    
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method= 'bfs', # 'astar', #'bfs', # TO CHANGE
            retriever_config= BFSSearchConfig(), #AStarGraphSearchConfig(), BFSSearchConfig(), # TO CHANGE
            cache_config=KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)), # TO CHANGE
        answer_generator_config=QALLMGeneratorConfig(lang='eng')),
    
    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(),
        updator_config=LLMUpdatorConfig()),
    
    log=Logger('log/main'))

#### 2. Инициализируем граф знаний

In [4]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

In [17]:
# ATTENTION !!!
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) -[r] -> () delete a, r")
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) delete a")
# ATTENTION !!!

#### 3. Добавляем в граф информацию

In [5]:
rkg_main.update_memory([
    "Mikhail Menshchikov is currently a second-year master's student at ITMO.",
    "Mikhail Menshchikov is studying in the Master's program 'Deep Learning and Generative AI'",
    "Mikhail Menshchikov completed his bachelor's degree at Petrozavodsk State University",
    "Petrozavodsk State University is where Mikhail Menshchikov received his bachelor's degree.",
    "Mikhail Menshchikov studied at Petrozavodsk State University and received a bachelor's degree."])

100%|██████████| 5/5 [00:34<00:00,  6.85s/it]


In [ ]:
rkg_main.kg_model.graph_struct.db_conn.triplets_ids['98bb0615a85f0eed2715d153297e6fd5']

Triplet(start_node=Node(name='Petrozavodsk State University', type=<NodeType.object: 'object'>, id='c0df0fdb2913db50f7c66f5ac9e58468', prop={}, stringified='Petrozavodsk State University'), relation=Relation(name='episodic', type=<RelationType.episodic: 'episodic'>, id='5276f0eabef68c998bac7867e9fb1d12', prop={}), end_node=Node(name="Mikhail Menshchikov completed his bachelor's degree at Petrozavodsk State University", type=<NodeType.episodic: 'episodic'>, id='b3553c268945cb3ccc9f38e38b32c296', prop={}, stringified="Mikhail Menshchikov completed his bachelor's degree at Petrozavodsk State University"), id='98bb0615a85f0eed2715d153297e6fd5', stringified="Mikhail Menshchikov completed his bachelor's degree at Petrozavodsk State University")

#### 4. Q&A

In [6]:
rkg_main.answer_question("What program is Mikhail Menshchikov studying for his master's degree?")

retrieved_triplets: 
Triplet(start_node=Node(name='studying', type='object', id='1', prop={}, stringified='studying'), relation=Relation(name='', type=<RelationType.hyper: 'hyper'>, id='1', prop={}), end_node=Node(name='Mikhail Menshchikov is studying', type='hyper', id='1', prop={}, stringified='Mikhail Menshchikov is studying'), id='d8b4d611a33eb493ef09b4da187d6893', stringified=None)
Triplet(start_node=Node(name='studying', type='object', id='1', prop={}, stringified='studying'), relation=Relation(name='', type=<RelationType.hyper: 'hyper'>, id='1', prop={}), end_node=Node(name="Mikhail Menshchikov is studying in the Master's program 'Deep Learning and Generative AI'", type='hyper', id='1', prop={}, stringified="Mikhail Menshchikov is studying in the Master's program 'Deep Learning and Generative AI'"), id='69700ddb97181c48cdfc3f5cb3730be8', stringified=None)
Triplet(start_node=Node(name='mikhail menshchikov', type=<NodeType.object: 'object'>, id='3b421655267a5bba2f9c5e944aacc1ac', 

'Deep Learning and Generative AI'

In [7]:
rkg_main.answer_question("Where did Mikhail Menshchikov receive his bachelor's degree?")

retrieved_triplets: 
Triplet(start_node=Node(name='mikhail menshchikov', type=<NodeType.object: 'object'>, id='3b421655267a5bba2f9c5e944aacc1ac', prop={}, stringified='mikhail menshchikov'), relation=Relation(name='', type=<RelationType.simple: 'simple'>, id='851c9f97a67be9fc57e62e77b85e8ffc', prop={}), end_node=Node(name="bachelor's degree", type=<NodeType.object: 'object'>, id='d5b0bfdf754fdc958fc77583faffb335', prop={}, stringified="bachelor's degree"), id='da0edf262964bcc2d4bca33f9e94ed96', stringified=None)
[None]


'Unknown'